# eICU Sepsis-3 Table Creation for BigQuery

This notebook creates all necessary tables for Sepsis-3 analysis using eICU database in BigQuery.

**Target Dataset:** `my-new-project-473015.my_eicu_derived`  
**Region:** US  
**Source:** eICU Collaborative Research Database from PhysioNet

## Dependencies Order:
1. vitalsign (from vitalPeriodic and vitalAperiodic)
2. weight_icustay (from patient table)
3. urine_output (from intakeOutput table)
4. urine_output_rate (depends on urine_output, weight_icustay)
5. ventilation (from respiratoryCare and apacheApsVar)
6. suspicion_of_infection (from medication and microlab)
7. sofa (depends on vitalsign, ventilation, urine_output_rate, lab, medication)
8. sepsis3 (depends on suspicion_of_infection, sofa)

## Key Differences from MIMIC-IV:
- eICU uses `patientunitstayid` instead of `stay_id`
- eICU uses `uniquepid` instead of `subject_id`
- Time in eICU is stored as offset from ICU admission (in minutes)
- Different table structure for vitals, labs, and medications

---

## Setup

**YOU SHOULD CHANGE YOUR PROJECT AND DATASET NAME**

In [1]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

# Initialize BigQuery client
client = bigquery.Client(project='my-new-project-473015')

# Dataset configuration
DATASET_ID = 'my_eicu_derived'
PROJECT_ID = 'my-new-project-473015'
SOURCE_DATASET = 'physionet-data.eicu_crd'  # eICU source dataset

print("="*70)
print(f"eICU Sepsis-3 Table Creation")
print(f"Target dataset: {PROJECT_ID}.{DATASET_ID}")
print(f"Source dataset: {SOURCE_DATASET}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

# Create dataset if it doesn't exist
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
try:
    client.get_dataset(dataset_ref)
    print(f"\n✓ Dataset {DATASET_ID} already exists")
except:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"  # Same region as eICU
    client.create_dataset(dataset)
    print(f"\n✓ Created dataset {DATASET_ID} in US region")

print("\n✅ Setup complete! Ready to create tables.")

eICU Sepsis-3 Table Creation
Target dataset: my-new-project-473015.my_eicu_derived
Source dataset: physionet-data.eicu_crd
Timestamp: 2026-01-08 20:17:23

✓ Dataset my_eicu_derived already exists

✅ Setup complete! Ready to create tables.


---

## STEP 1: Create Vitalsign Table

This table extracts and pivots vital signs data from eICU vitalPeriodic and vitalAperiodic tables.

**Contains:**
- Heart rate, blood pressure (systolic/diastolic/mean)
- Respiratory rate
- SpO2 (oxygen saturation)
- Temperature (in Celsius)

**Source:** `vitalPeriodic` and `vitalAperiodic` tables  
**One row per:** patientunitstayid and observationoffset  

**Key eICU Variables:**
- vitalPeriodic: heartrate, systemicsystolic, systemicdiastolic, systemicmean, respiration, sao2, temperature
- vitalAperiodic: noninvasivesystolic, noninvasivediastolic, noninvasivemean

In [2]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.vitalsign` AS
WITH vital_periodic AS (
    SELECT
        patientunitstayid
        , observationoffset
        , CASE
            WHEN heartrate > 0 AND heartrate < 300 THEN heartrate
            ELSE NULL
          END AS heart_rate
        , CASE
            WHEN systemicsystolic > 0 AND systemicsystolic < 400 THEN systemicsystolic
            ELSE NULL
          END AS sbp
        , CASE
            WHEN systemicdiastolic > 0 AND systemicdiastolic < 300 THEN systemicdiastolic
            ELSE NULL
          END AS dbp
        , CASE
            WHEN systemicmean > 0 AND systemicmean < 300 THEN systemicmean
            ELSE NULL
          END AS mbp
        , CASE
            WHEN respiration > 0 AND respiration < 70 THEN respiration
            ELSE NULL
          END AS resp_rate
        , CASE
            WHEN sao2 > 0 AND sao2 <= 100 THEN sao2
            ELSE NULL
          END AS spo2
        , CASE
            -- Temperature in eICU is in Celsius
            WHEN temperature > 10 AND temperature < 50 THEN temperature
            ELSE NULL
          END AS temperature
    FROM `physionet-data.eicu_crd.vitalperiodic`
)
, vital_aperiodic AS (
    SELECT
        patientunitstayid
        , observationoffset
        , CASE
            WHEN noninvasivesystolic > 0 AND noninvasivesystolic < 400 THEN noninvasivesystolic
            ELSE NULL
          END AS sbp_ni
        , CASE
            WHEN noninvasivediastolic > 0 AND noninvasivediastolic < 300 THEN noninvasivediastolic
            ELSE NULL
          END AS dbp_ni
        , CASE
            WHEN noninvasivemean > 0 AND noninvasivemean < 300 THEN noninvasivemean
            ELSE NULL
          END AS mbp_ni
    FROM `physionet-data.eicu_crd.vitalaperiodic`
)
, combined AS (
    SELECT
        COALESCE(vp.patientunitstayid, va.patientunitstayid) AS patientunitstayid
        , COALESCE(vp.observationoffset, va.observationoffset) AS observationoffset
        , vp.heart_rate
        , COALESCE(vp.sbp, va.sbp_ni) AS sbp
        , COALESCE(vp.dbp, va.dbp_ni) AS dbp
        , COALESCE(vp.mbp, va.mbp_ni) AS mbp
        , va.sbp_ni
        , va.dbp_ni
        , va.mbp_ni
        , vp.resp_rate
        , vp.spo2
        , vp.temperature
    FROM vital_periodic vp
    FULL OUTER JOIN vital_aperiodic va
        ON vp.patientunitstayid = va.patientunitstayid
        AND vp.observationoffset = va.observationoffset
)
SELECT
    patientunitstayid
    , observationoffset
    , AVG(heart_rate) AS heart_rate
    , AVG(sbp) AS sbp
    , AVG(dbp) AS dbp
    , AVG(mbp) AS mbp
    , AVG(sbp_ni) AS sbp_ni
    , AVG(dbp_ni) AS dbp_ni
    , AVG(mbp_ni) AS mbp_ni
    , AVG(resp_rate) AS resp_rate
    , AVG(spo2) AS spo2
    , ROUND(AVG(temperature), 2) AS temperature
FROM combined
GROUP BY patientunitstayid, observationoffset;

Query is running:   0%|          |

""


In [3]:
%%bigquery

-- Verification: Check vitalsign table
SELECT
    'vitalsign' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT patientunitstayid) as unique_icu_stays
FROM `my-new-project-473015.my_eicu_derived.vitalsign`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays
0,vitalsign,166376651,193221


---

## STEP 2: Create Weight Table

This table extracts patient weight from the patient table.

**Contains:**
- Admission weight

**Purpose:** Used to calculate weight-adjusted metrics (e.g., urine output per kg)

**Source:** `patient` table  
**Note:** eICU stores weight directly in the patient table as `admissionweight`

In [4]:
%%bigquery

-- Recreate weight_icustay using pivoted_weight
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.weight_icustay` AS
WITH weight_ranked AS (
    SELECT
        patientunitstayid
        , chartoffset
        , weight
        , weight_type
        , source_table
        -- Prioritize: admission weight > daily weight > other
        , ROW_NUMBER() OVER (
            PARTITION BY patientunitstayid
            ORDER BY
                CASE
                    WHEN LOWER(weight_type) LIKE '%admit%' THEN 1
                    WHEN LOWER(weight_type) LIKE '%daily%' THEN 2
                    ELSE 3
                END,
                ABS(chartoffset)  -- Closest to ICU admission
        ) AS rn
    FROM `physionet-data.eicu_crd_derived.pivoted_weight`
    WHERE weight IS NOT NULL
      AND weight > 10    -- Reasonable minimum (kg)
      AND weight < 500   -- Reasonable maximum (kg)
)
SELECT
    patientunitstayid
    , weight
    , weight_type
    , source_table
    , chartoffset AS weight_offset
FROM weight_ranked
WHERE rn = 1;

Query is running:   0%|          |

""


In [5]:
%%bigquery

-- Verification: Check weight table
SELECT
    'weight_icustay' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT patientunitstayid) as unique_icu_stays,
    ROUND(AVG(weight), 1) as avg_weight,
    MIN(weight) as min_weight,
    MAX(weight) as max_weight
FROM `my-new-project-473015.my_eicu_derived.weight_icustay`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_weight,min_weight,max_weight
0,weight_icustay,190650,190650,83.9,10.12,456.0


---

## STEP 3: Create Urine Output Table

This table extracts urine output measurements from the intakeOutput table.

**Contains:**
- Urine output volumes
- Time offsets

**Source:** `intakeOutput` table  
**Filter:** celllabel LIKE '%Urine%' OR cellpath LIKE '%Output%Urine%'

In [6]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.urine_output` AS
SELECT
    patientunitstayid
    , intakeoutputoffset AS chartoffset
    , CASE
        WHEN cellvaluenumeric > 0 THEN cellvaluenumeric
        ELSE 0
      END AS urineoutput
FROM `physionet-data.eicu_crd.intakeoutput`
WHERE (
    LOWER(celllabel) LIKE '%urine%'
    OR LOWER(cellpath) LIKE '%output%urine%'
    OR LOWER(celllabel) LIKE '%foley%'
    OR LOWER(celllabel) LIKE '%void%'
)
AND cellvaluenumeric IS NOT NULL
AND cellvaluenumeric >= 0
AND cellvaluenumeric < 10000  -- Exclude unrealistic values

Query is running:   0%|          |

""


In [7]:
%%bigquery

-- Verification: Check urine_output table
SELECT
    'urine_output' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT patientunitstayid) as unique_icu_stays,
    ROUND(AVG(urineoutput), 1) as avg_uo,
    MIN(urineoutput) as min_uo,
    MAX(urineoutput) as max_uo
FROM `my-new-project-473015.my_eicu_derived.urine_output`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_uo,min_uo,max_uo
0,urine_output,3796397,153977,249.1,0.0,9999.0


---

## STEP 4: Create Urine Output Rate Table

This table calculates urine output rates per hour over 24-hour windows.

**Contains:**
- Urine output over 24 hour windows
- Weight-adjusted urine output (ml/kg/hr)

**Purpose:** Used for SOFA renal component calculation (oliguria detection)

**Dependencies:** `urine_output` and `weight_icustay` tables

In [8]:
%%bigquery

-- Recreate urine_output_rate using pivoted_weight
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.urine_output_rate` AS
WITH uo_with_weight AS (
    SELECT
        uo.patientunitstayid
        , uo.chartoffset
        , uo.urineoutput
        , wt.weight
    FROM `my-new-project-473015.my_eicu_derived.urine_output` uo
    LEFT JOIN `my-new-project-473015.my_eicu_derived.weight_icustay` wt
        ON uo.patientunitstayid = wt.patientunitstayid
)
, uo_rolling AS (
    SELECT
        patientunitstayid
        , chartoffset
        , weight
        , urineoutput
        -- 24-hour rolling sum using RANGE window
        , SUM(urineoutput) OVER (
            PARTITION BY patientunitstayid
            ORDER BY chartoffset
            RANGE BETWEEN 1440 PRECEDING AND CURRENT ROW
        ) AS uo_24hr
        -- First measurement time in 24hr window
        , chartoffset - MIN(chartoffset) OVER (
            PARTITION BY patientunitstayid
            ORDER BY chartoffset
            RANGE BETWEEN 1440 PRECEDING AND CURRENT ROW
        ) AS uo_tm_24hr_minutes
    FROM uo_with_weight
)
SELECT
    patientunitstayid
    , chartoffset
    , weight
    , urineoutput
    , uo_24hr
    , uo_tm_24hr_minutes / 60.0 AS uo_tm_24hr  -- Convert to hours
    -- Weight-adjusted urine output rate (ml/kg/hr)
    , CASE
        WHEN weight > 0 AND weight IS NOT NULL
        THEN ROUND(uo_24hr / weight / 24.0, 2)
        ELSE NULL
      END AS uo_mlkghr_24hr
FROM uo_rolling;

Query is running:   0%|          |

""


In [9]:
%%bigquery

-- Verification: Check urine_output_rate table
SELECT
    'urine_output_rate' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT patientunitstayid) as unique_icu_stays,
    ROUND(AVG(uo_24hr), 1) as avg_uo_24hr,
    ROUND(AVG(uo_mlkghr_24hr), 2) as avg_uo_mlkghr
FROM `my-new-project-473015.my_eicu_derived.urine_output_rate`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_uo_24hr,avg_uo_mlkghr
0,urine_output_rate,3796397,153977,1984.4,1.03


---

## STEP 5: Create Ventilation Table

This table classifies ventilation status from respiratoryCare and apacheApsVar tables.

**Categories:**
1. **InvasiveVent** - Invasive mechanical ventilation
2. **NonInvasiveVent** - Non-invasive ventilation (BiPAP, CPAP)
3. **SupplementalOxygen** - Supplemental oxygen only
4. **None** - No respiratory support

**Source:** `respiratoryCare`, `respiratoryCharting`, `apacheApsVar` tables

In [10]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.ventilation` AS
WITH vent_rc AS (
    -- From respiratoryCare table
    SELECT
        patientunitstayid
        , respcarestatusoffset AS chartoffset
        , CASE
            WHEN LOWER(airwaytype) LIKE '%endotracheal%'
                OR LOWER(airwaytype) LIKE '%oral%ett%'
                OR LOWER(airwaytype) LIKE '%nasal%ett%'
                THEN 'InvasiveVent'
            WHEN LOWER(airwaytype) LIKE '%trach%'
                THEN 'Tracheostomy'
            WHEN LOWER(airwaytype) LIKE '%bipap%'
                OR LOWER(airwaytype) LIKE '%cpap%'
                OR LOWER(airwaytype) LIKE '%niv%'
                THEN 'NonInvasiveVent'
            ELSE NULL
          END AS ventilation_status
    FROM `physionet-data.eicu_crd.respiratorycare`
    WHERE airwaytype IS NOT NULL
)
, vent_chart AS (
    -- From respiratoryCharting table
    SELECT
        patientunitstayid
        , respchartoffset AS chartoffset
        , CASE
            WHEN LOWER(respcharttypecat) LIKE '%vent%'
                AND (LOWER(respchartvaluelabel) LIKE '%mode%'
                     OR LOWER(respchartvaluelabel) LIKE '%setting%')
                THEN 'InvasiveVent'
            WHEN LOWER(respchartvaluelabel) LIKE '%bipap%'
                OR LOWER(respchartvaluelabel) LIKE '%cpap%'
                THEN 'NonInvasiveVent'
            WHEN LOWER(respchartvaluelabel) LIKE '%nasal cannula%'
                OR LOWER(respchartvaluelabel) LIKE '%face mask%'
                OR LOWER(respchartvaluelabel) LIKE '%high flow%'
                THEN 'SupplementalOxygen'
            ELSE NULL
          END AS ventilation_status
    FROM `physionet-data.eicu_crd.respiratorycharting`
    WHERE respchartvaluelabel IS NOT NULL
)
, vent_apache AS (
    -- From apacheApsVar: intubated flag
    SELECT
        patientunitstayid
        , 0 AS chartoffset  -- APACHE is for first 24 hours
        , CASE
            WHEN intubated = 1 THEN 'InvasiveVent'
            ELSE NULL
          END AS ventilation_status
    FROM `physionet-data.eicu_crd.apacheapsvar`
    WHERE intubated = 1
)
, all_vent AS (
    SELECT patientunitstayid, chartoffset, ventilation_status FROM vent_rc WHERE ventilation_status IS NOT NULL
    UNION ALL
    SELECT patientunitstayid, chartoffset, ventilation_status FROM vent_chart WHERE ventilation_status IS NOT NULL
    UNION ALL
    SELECT patientunitstayid, chartoffset, ventilation_status FROM vent_apache WHERE ventilation_status IS NOT NULL
)
SELECT
    patientunitstayid
    , chartoffset
    -- Priority: Tracheostomy > InvasiveVent > NonInvasiveVent > SupplementalOxygen
    , CASE
        WHEN MAX(CASE WHEN ventilation_status = 'Tracheostomy' THEN 1 ELSE 0 END) = 1 THEN 'Tracheostomy'
        WHEN MAX(CASE WHEN ventilation_status = 'InvasiveVent' THEN 1 ELSE 0 END) = 1 THEN 'InvasiveVent'
        WHEN MAX(CASE WHEN ventilation_status = 'NonInvasiveVent' THEN 1 ELSE 0 END) = 1 THEN 'NonInvasiveVent'
        WHEN MAX(CASE WHEN ventilation_status = 'SupplementalOxygen' THEN 1 ELSE 0 END) = 1 THEN 'SupplementalOxygen'
        ELSE 'None'
      END AS ventilation_status
FROM all_vent
GROUP BY patientunitstayid, chartoffset;

Query is running:   0%|          |

""


In [11]:
%%bigquery

-- Verification: Check ventilation table
SELECT
    ventilation_status,
    COUNT(*) as count,
    COUNT(DISTINCT patientunitstayid) as unique_icu_stays
FROM `my-new-project-473015.my_eicu_derived.ventilation`
GROUP BY ventilation_status
ORDER BY count DESC;

Query is running:   0%|          |

Downloading:   0%|          |

,ventilation_status,count,unique_icu_stays
0,NonInvasiveVent,152604,8031
1,InvasiveVent,84937,28771
2,Tracheostomy,8308,1028


---

## STEP 6: Create Suspicion of Infection Table

This table identifies suspected infections based on:
1. Antibiotic administration
2. Culture orders (microbiological samples)

**Sepsis-3 Definition:**
- Antibiotic started within 72 hours of culture, OR
- Culture taken within 24 hours before antibiotic start

**Source:** `medication` and `microlab` tables

In [12]:
%%bigquery

-- Top 50 antibiotics in eICU medication table (using MIMIC-IV official list)
SELECT
    drugname,
    routeadmin,
    COUNT(*) as prescription_count,
    COUNT(DISTINCT patientunitstayid) as patient_count
FROM `physionet-data.eicu_crd.medication`
WHERE (
    -- MIMIC-IV official antibiotic list
    LOWER(drugname) LIKE '%adoxa%'
    OR LOWER(drugname) LIKE '%ala-tet%'
    OR LOWER(drugname) LIKE '%alodox%'
    OR LOWER(drugname) LIKE '%amikacin%'
    OR LOWER(drugname) LIKE '%amikin%'
    OR LOWER(drugname) LIKE '%amoxicill%'
    OR LOWER(drugname) LIKE '%amphotericin%'
    OR LOWER(drugname) LIKE '%anidulafungin%'
    OR LOWER(drugname) LIKE '%ancef%'
    OR LOWER(drugname) LIKE '%clavulanate%'
    OR LOWER(drugname) LIKE '%ampicillin%'
    OR LOWER(drugname) LIKE '%augmentin%'
    OR LOWER(drugname) LIKE '%avelox%'
    OR LOWER(drugname) LIKE '%avidoxy%'
    OR LOWER(drugname) LIKE '%azactam%'
    OR LOWER(drugname) LIKE '%azithromycin%'
    OR LOWER(drugname) LIKE '%aztreonam%'
    OR LOWER(drugname) LIKE '%axetil%'
    OR LOWER(drugname) LIKE '%bactocill%'
    OR LOWER(drugname) LIKE '%bactrim%'
    OR LOWER(drugname) LIKE '%bactroban%'
    OR LOWER(drugname) LIKE '%bethkis%'
    OR LOWER(drugname) LIKE '%biaxin%'
    OR LOWER(drugname) LIKE '%bicillin l-a%'
    OR LOWER(drugname) LIKE '%cayston%'
    OR LOWER(drugname) LIKE '%cefazolin%'
    OR LOWER(drugname) LIKE '%cedax%'
    OR LOWER(drugname) LIKE '%cefoxitin%'
    OR LOWER(drugname) LIKE '%ceftazidime%'
    OR LOWER(drugname) LIKE '%cefaclor%'
    OR LOWER(drugname) LIKE '%cefadroxil%'
    OR LOWER(drugname) LIKE '%cefdinir%'
    OR LOWER(drugname) LIKE '%cefditoren%'
    OR LOWER(drugname) LIKE '%cefepime%'
    OR LOWER(drugname) LIKE '%cefotan%'
    OR LOWER(drugname) LIKE '%cefotetan%'
    OR LOWER(drugname) LIKE '%cefotaxime%'
    OR LOWER(drugname) LIKE '%ceftaroline%'
    OR LOWER(drugname) LIKE '%cefpodoxime%'
    OR LOWER(drugname) LIKE '%cefpirome%'
    OR LOWER(drugname) LIKE '%cefprozil%'
    OR LOWER(drugname) LIKE '%ceftibuten%'
    OR LOWER(drugname) LIKE '%ceftin%'
    OR LOWER(drugname) LIKE '%ceftriaxone%'
    OR LOWER(drugname) LIKE '%cefuroxime%'
    OR LOWER(drugname) LIKE '%cephalexin%'
    OR LOWER(drugname) LIKE '%cephalothin%'
    OR LOWER(drugname) LIKE '%cephapririn%'
    OR LOWER(drugname) LIKE '%chloramphenicol%'
    OR LOWER(drugname) LIKE '%cipro%'
    OR LOWER(drugname) LIKE '%ciprofloxacin%'
    OR LOWER(drugname) LIKE '%claforan%'
    OR LOWER(drugname) LIKE '%clarithromycin%'
    OR LOWER(drugname) LIKE '%cleocin%'
    OR LOWER(drugname) LIKE '%clindamycin%'
    OR LOWER(drugname) LIKE '%cubicin%'
    OR LOWER(drugname) LIKE '%dicloxacillin%'
    OR LOWER(drugname) LIKE '%dirithromycin%'
    OR LOWER(drugname) LIKE '%doryx%'
    OR LOWER(drugname) LIKE '%doxycy%'
    OR LOWER(drugname) LIKE '%duricef%'
    OR LOWER(drugname) LIKE '%dynacin%'
    OR LOWER(drugname) LIKE '%ery-tab%'
    OR LOWER(drugname) LIKE '%eryped%'
    OR LOWER(drugname) LIKE '%eryc%'
    OR LOWER(drugname) LIKE '%erythrocin%'
    OR LOWER(drugname) LIKE '%erythromycin%'
    OR LOWER(drugname) LIKE '%factive%'
    OR LOWER(drugname) LIKE '%flagyl%'
    OR LOWER(drugname) LIKE '%fortaz%'
    OR LOWER(drugname) LIKE '%furadantin%'
    OR LOWER(drugname) LIKE '%garamycin%'
    OR LOWER(drugname) LIKE '%gentamicin%'
    OR LOWER(drugname) LIKE '%kanamycin%'
    OR LOWER(drugname) LIKE '%keflex%'
    OR LOWER(drugname) LIKE '%kefzol%'
    OR LOWER(drugname) LIKE '%ketek%'
    OR LOWER(drugname) LIKE '%levaquin%'
    OR LOWER(drugname) LIKE '%levofloxacin%'
    OR LOWER(drugname) LIKE '%lincocin%'
    OR LOWER(drugname) LIKE '%linezolid%'
    OR LOWER(drugname) LIKE '%macrobid%'
    OR LOWER(drugname) LIKE '%macrodantin%'
    OR LOWER(drugname) LIKE '%maxipime%'
    OR LOWER(drugname) LIKE '%mefoxin%'
    OR LOWER(drugname) LIKE '%metronidazole%'
    OR LOWER(drugname) LIKE '%meropenem%'
    OR LOWER(drugname) LIKE '%methicillin%'
    OR LOWER(drugname) LIKE '%minocin%'
    OR LOWER(drugname) LIKE '%minocycline%'
    OR LOWER(drugname) LIKE '%monodox%'
    OR LOWER(drugname) LIKE '%monurol%'
    OR LOWER(drugname) LIKE '%morgidox%'
    OR LOWER(drugname) LIKE '%moxatag%'
    OR LOWER(drugname) LIKE '%moxifloxacin%'
    OR LOWER(drugname) LIKE '%mupirocin%'
    OR LOWER(drugname) LIKE '%myrac%'
    OR LOWER(drugname) LIKE '%nafcillin%'
    OR LOWER(drugname) LIKE '%neomycin%'
    OR LOWER(drugname) LIKE '%nicazel doxy 30%'
    OR LOWER(drugname) LIKE '%nitrofurantoin%'
    OR LOWER(drugname) LIKE '%norfloxacin%'
    OR LOWER(drugname) LIKE '%noroxin%'
    OR LOWER(drugname) LIKE '%ocudox%'
    OR LOWER(drugname) LIKE '%ofloxacin%'
    OR LOWER(drugname) LIKE '%omnicef%'
    OR LOWER(drugname) LIKE '%oracea%'
    OR LOWER(drugname) LIKE '%oraxyl%'
    OR LOWER(drugname) LIKE '%oxacillin%'
    OR LOWER(drugname) LIKE '%pc pen vk%'
    OR LOWER(drugname) LIKE '%pce dispertab%'
    OR LOWER(drugname) LIKE '%panixine%'
    OR LOWER(drugname) LIKE '%pediazole%'
    OR LOWER(drugname) LIKE '%penicillin%'
    OR LOWER(drugname) LIKE '%periostat%'
    OR LOWER(drugname) LIKE '%pfizerpen%'
    OR LOWER(drugname) LIKE '%piperacillin%'
    OR LOWER(drugname) LIKE '%tazobactam%'
    OR LOWER(drugname) LIKE '%primsol%'
    OR LOWER(drugname) LIKE '%proquin%'
    OR LOWER(drugname) LIKE '%raniclor%'
    OR LOWER(drugname) LIKE '%rifadin%'
    OR LOWER(drugname) LIKE '%rifampin%'
    OR LOWER(drugname) LIKE '%rocephin%'
    OR LOWER(drugname) LIKE '%smz-tmp%'
    OR LOWER(drugname) LIKE '%septra%'
    OR LOWER(drugname) LIKE '%solodyn%'
    OR LOWER(drugname) LIKE '%spectracef%'
    OR LOWER(drugname) LIKE '%streptomycin%'
    OR LOWER(drugname) LIKE '%sulfadiazine%'
    OR LOWER(drugname) LIKE '%sulfamethoxazole%'
    OR LOWER(drugname) LIKE '%trimethoprim%'
    OR LOWER(drugname) LIKE '%sulfatrim%'
    OR LOWER(drugname) LIKE '%sulfisoxazole%'
    OR LOWER(drugname) LIKE '%suprax%'
    OR LOWER(drugname) LIKE '%synercid%'
    OR LOWER(drugname) LIKE '%tazicef%'
    OR LOWER(drugname) LIKE '%tetracycline%'
    OR LOWER(drugname) LIKE '%timentin%'
    OR LOWER(drugname) LIKE '%tobramycin%'
    OR LOWER(drugname) LIKE '%unasyn%'
    OR LOWER(drugname) LIKE '%vancocin%'
    OR LOWER(drugname) LIKE '%vancomycin%'
    OR LOWER(drugname) LIKE '%vantin%'
    OR LOWER(drugname) LIKE '%vibativ%'
    OR LOWER(drugname) LIKE '%vibra-tabs%'
    OR LOWER(drugname) LIKE '%vibramycin%'
    OR LOWER(drugname) LIKE '%zinacef%'
    OR LOWER(drugname) LIKE '%zithromax%'
    OR LOWER(drugname) LIKE '%zosyn%'
    OR LOWER(drugname) LIKE '%zyvox%'
)
-- Exclude topical formulations
AND LOWER(drugname) NOT LIKE '%cream%'
AND LOWER(drugname) NOT LIKE '%ointment%'
AND LOWER(drugname) NOT LIKE '%ophthalmic%'
AND LOWER(drugname) NOT LIKE '%otic%'
AND LOWER(drugname) NOT LIKE '%eye%'
AND LOWER(drugname) NOT LIKE '%ear drop%'
AND LOWER(drugname) NOT LIKE '%gel%'
AND LOWER(drugname) NOT LIKE '%desensitization%'
AND drugstartoffset IS NOT NULL
GROUP BY drugname, routeadmin
ORDER BY prescription_count DESC
LIMIT 50;

Query is running:   0%|          |

Downloading:   0%|          |

,drugname,routeadmin,prescription_count,patient_count
0,VANCOMYCIN HCL 1000 MG IV SOLR,IV,6818,3878
1,cefepime,MISC,6473,814
2,ZOSYN,IV,5900,3853
3,VANCOCIN,IV,5391,3206
4,LEVAQUIN,IV,4817,3355
5,piperacillin-tazobactam,MISC,4296,421
6,ANCEF,IV,4110,2355
7,VANCOmycin 1 GM in NS 250 mL IVPB,IVPB,3813,2799
8,CEFAZOLIN 2 GM IN NS 100 ML IVPB (REPACKAGE),IV,3454,2347
9,LEVOFLOXACIN IN D5W 750 MG/150ML IV SOLN,IV,3239,2290


In [13]:
%%bigquery

-- Updated suspicion_of_infection table using MIMIC-IV official antibiotic list
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.suspicion_of_infection` AS
WITH antibiotics AS (
    -- Extract antibiotic medications using MIMIC-IV official list
    SELECT
        patientunitstayid
        , drugstartoffset AS antibiotic_offset
        , drugname AS antibiotic
        , routeadmin
    FROM `physionet-data.eicu_crd.medication`
    WHERE (
        -- MIMIC-IV official antibiotic list (from antibiotic.sql)
        -- Generic names and brand names
        LOWER(drugname) LIKE '%adoxa%'
        OR LOWER(drugname) LIKE '%ala-tet%'
        OR LOWER(drugname) LIKE '%alodox%'
        OR LOWER(drugname) LIKE '%amikacin%'
        OR LOWER(drugname) LIKE '%amikin%'
        OR LOWER(drugname) LIKE '%amoxicill%'
        OR LOWER(drugname) LIKE '%amphotericin%'
        OR LOWER(drugname) LIKE '%anidulafungin%'
        OR LOWER(drugname) LIKE '%ancef%'
        OR LOWER(drugname) LIKE '%clavulanate%'
        OR LOWER(drugname) LIKE '%ampicillin%'
        OR LOWER(drugname) LIKE '%augmentin%'
        OR LOWER(drugname) LIKE '%avelox%'
        OR LOWER(drugname) LIKE '%avidoxy%'
        OR LOWER(drugname) LIKE '%azactam%'
        OR LOWER(drugname) LIKE '%azithromycin%'
        OR LOWER(drugname) LIKE '%aztreonam%'
        OR LOWER(drugname) LIKE '%axetil%'
        OR LOWER(drugname) LIKE '%bactocill%'
        OR LOWER(drugname) LIKE '%bactrim%'
        OR LOWER(drugname) LIKE '%bactroban%'
        OR LOWER(drugname) LIKE '%bethkis%'
        OR LOWER(drugname) LIKE '%biaxin%'
        OR LOWER(drugname) LIKE '%bicillin l-a%'
        OR LOWER(drugname) LIKE '%cayston%'
        OR LOWER(drugname) LIKE '%cefazolin%'
        OR LOWER(drugname) LIKE '%cedax%'
        OR LOWER(drugname) LIKE '%cefoxitin%'
        OR LOWER(drugname) LIKE '%ceftazidime%'
        OR LOWER(drugname) LIKE '%cefaclor%'
        OR LOWER(drugname) LIKE '%cefadroxil%'
        OR LOWER(drugname) LIKE '%cefdinir%'
        OR LOWER(drugname) LIKE '%cefditoren%'
        OR LOWER(drugname) LIKE '%cefepime%'
        OR LOWER(drugname) LIKE '%cefotan%'
        OR LOWER(drugname) LIKE '%cefotetan%'
        OR LOWER(drugname) LIKE '%cefotaxime%'
        OR LOWER(drugname) LIKE '%ceftaroline%'
        OR LOWER(drugname) LIKE '%cefpodoxime%'
        OR LOWER(drugname) LIKE '%cefpirome%'
        OR LOWER(drugname) LIKE '%cefprozil%'
        OR LOWER(drugname) LIKE '%ceftibuten%'
        OR LOWER(drugname) LIKE '%ceftin%'
        OR LOWER(drugname) LIKE '%ceftriaxone%'
        OR LOWER(drugname) LIKE '%cefuroxime%'
        OR LOWER(drugname) LIKE '%cephalexin%'
        OR LOWER(drugname) LIKE '%cephalothin%'
        OR LOWER(drugname) LIKE '%cephapririn%'
        OR LOWER(drugname) LIKE '%chloramphenicol%'
        OR LOWER(drugname) LIKE '%cipro%'
        OR LOWER(drugname) LIKE '%ciprofloxacin%'
        OR LOWER(drugname) LIKE '%claforan%'
        OR LOWER(drugname) LIKE '%clarithromycin%'
        OR LOWER(drugname) LIKE '%cleocin%'
        OR LOWER(drugname) LIKE '%clindamycin%'
        OR LOWER(drugname) LIKE '%cubicin%'
        OR LOWER(drugname) LIKE '%dicloxacillin%'
        OR LOWER(drugname) LIKE '%dirithromycin%'
        OR LOWER(drugname) LIKE '%doryx%'
        OR LOWER(drugname) LIKE '%doxycy%'
        OR LOWER(drugname) LIKE '%duricef%'
        OR LOWER(drugname) LIKE '%dynacin%'
        OR LOWER(drugname) LIKE '%ery-tab%'
        OR LOWER(drugname) LIKE '%eryped%'
        OR LOWER(drugname) LIKE '%eryc%'
        OR LOWER(drugname) LIKE '%erythrocin%'
        OR LOWER(drugname) LIKE '%erythromycin%'
        OR LOWER(drugname) LIKE '%factive%'
        OR LOWER(drugname) LIKE '%flagyl%'
        OR LOWER(drugname) LIKE '%fortaz%'
        OR LOWER(drugname) LIKE '%furadantin%'
        OR LOWER(drugname) LIKE '%garamycin%'
        OR LOWER(drugname) LIKE '%gentamicin%'
        OR LOWER(drugname) LIKE '%kanamycin%'
        OR LOWER(drugname) LIKE '%keflex%'
        OR LOWER(drugname) LIKE '%kefzol%'
        OR LOWER(drugname) LIKE '%ketek%'
        OR LOWER(drugname) LIKE '%levaquin%'
        OR LOWER(drugname) LIKE '%levofloxacin%'
        OR LOWER(drugname) LIKE '%lincocin%'
        OR LOWER(drugname) LIKE '%linezolid%'
        OR LOWER(drugname) LIKE '%macrobid%'
        OR LOWER(drugname) LIKE '%macrodantin%'
        OR LOWER(drugname) LIKE '%maxipime%'
        OR LOWER(drugname) LIKE '%mefoxin%'
        OR LOWER(drugname) LIKE '%metronidazole%'
        OR LOWER(drugname) LIKE '%meropenem%'
        OR LOWER(drugname) LIKE '%methicillin%'
        OR LOWER(drugname) LIKE '%minocin%'
        OR LOWER(drugname) LIKE '%minocycline%'
        OR LOWER(drugname) LIKE '%monodox%'
        OR LOWER(drugname) LIKE '%monurol%'
        OR LOWER(drugname) LIKE '%morgidox%'
        OR LOWER(drugname) LIKE '%moxatag%'
        OR LOWER(drugname) LIKE '%moxifloxacin%'
        OR LOWER(drugname) LIKE '%mupirocin%'
        OR LOWER(drugname) LIKE '%myrac%'
        OR LOWER(drugname) LIKE '%nafcillin%'
        OR LOWER(drugname) LIKE '%neomycin%'
        OR LOWER(drugname) LIKE '%nicazel doxy 30%'
        OR LOWER(drugname) LIKE '%nitrofurantoin%'
        OR LOWER(drugname) LIKE '%norfloxacin%'
        OR LOWER(drugname) LIKE '%noroxin%'
        OR LOWER(drugname) LIKE '%ocudox%'
        OR LOWER(drugname) LIKE '%ofloxacin%'
        OR LOWER(drugname) LIKE '%omnicef%'
        OR LOWER(drugname) LIKE '%oracea%'
        OR LOWER(drugname) LIKE '%oraxyl%'
        OR LOWER(drugname) LIKE '%oxacillin%'
        OR LOWER(drugname) LIKE '%pc pen vk%'
        OR LOWER(drugname) LIKE '%pce dispertab%'
        OR LOWER(drugname) LIKE '%panixine%'
        OR LOWER(drugname) LIKE '%pediazole%'
        OR LOWER(drugname) LIKE '%penicillin%'
        OR LOWER(drugname) LIKE '%periostat%'
        OR LOWER(drugname) LIKE '%pfizerpen%'
        OR LOWER(drugname) LIKE '%piperacillin%'
        OR LOWER(drugname) LIKE '%tazobactam%'
        OR LOWER(drugname) LIKE '%primsol%'
        OR LOWER(drugname) LIKE '%proquin%'
        OR LOWER(drugname) LIKE '%raniclor%'
        OR LOWER(drugname) LIKE '%rifadin%'
        OR LOWER(drugname) LIKE '%rifampin%'
        OR LOWER(drugname) LIKE '%rocephin%'
        OR LOWER(drugname) LIKE '%smz-tmp%'
        OR LOWER(drugname) LIKE '%septra%'
        OR LOWER(drugname) LIKE '%solodyn%'
        OR LOWER(drugname) LIKE '%spectracef%'
        OR LOWER(drugname) LIKE '%streptomycin%'
        OR LOWER(drugname) LIKE '%sulfadiazine%'
        OR LOWER(drugname) LIKE '%sulfamethoxazole%'
        OR LOWER(drugname) LIKE '%trimethoprim%'
        OR LOWER(drugname) LIKE '%sulfatrim%'
        OR LOWER(drugname) LIKE '%sulfisoxazole%'
        OR LOWER(drugname) LIKE '%suprax%'
        OR LOWER(drugname) LIKE '%synercid%'
        OR LOWER(drugname) LIKE '%tazicef%'
        OR LOWER(drugname) LIKE '%tetracycline%'
        OR LOWER(drugname) LIKE '%timentin%'
        OR LOWER(drugname) LIKE '%tobramycin%'
        OR LOWER(drugname) LIKE '%unasyn%'
        OR LOWER(drugname) LIKE '%vancocin%'
        OR LOWER(drugname) LIKE '%vancomycin%'
        OR LOWER(drugname) LIKE '%vantin%'
        OR LOWER(drugname) LIKE '%vibativ%'
        OR LOWER(drugname) LIKE '%vibra-tabs%'
        OR LOWER(drugname) LIKE '%vibramycin%'
        OR LOWER(drugname) LIKE '%zinacef%'
        OR LOWER(drugname) LIKE '%zithromax%'
        OR LOWER(drugname) LIKE '%zosyn%'
        OR LOWER(drugname) LIKE '%zyvox%'
    )
    -- Exclude topical, ophthalmic, otic routes (following MIMIC-IV logic)
    AND (routeadmin IS NULL
         OR LOWER(routeadmin) NOT IN ('topical', 'ophthalmic', 'otic', 'external', 'tp', 'top'))
    -- Exclude topical formulations
    AND LOWER(drugname) NOT LIKE '%cream%'
    AND LOWER(drugname) NOT LIKE '%ointment%'
    AND LOWER(drugname) NOT LIKE '%ophthalmic%'
    AND LOWER(drugname) NOT LIKE '%otic%'
    AND LOWER(drugname) NOT LIKE '%eye%'
    AND LOWER(drugname) NOT LIKE '%ear drop%'
    AND LOWER(drugname) NOT LIKE '%gel%'
    AND LOWER(drugname) NOT LIKE '%desensitization%'
    AND drugstartoffset IS NOT NULL
)
, cultures AS (
    -- Extract microbiology cultures
    SELECT
        patientunitstayid
        , culturetakenoffset AS culture_offset
        , culturesite AS specimen
        , organism
        , CASE WHEN organism IS NOT NULL AND organism != '' THEN 1 ELSE 0 END AS positive_culture
    FROM `physionet-data.eicu_crd.microlab`
    WHERE culturetakenoffset IS NOT NULL
)
, ab_tbl AS (
    SELECT
        patientunitstayid
        , antibiotic_offset
        , antibiotic
        , ROW_NUMBER() OVER (PARTITION BY patientunitstayid ORDER BY antibiotic_offset) AS ab_id
    FROM antibiotics
)
, me_tbl AS (
    SELECT
        patientunitstayid
        , culture_offset
        , specimen
        , positive_culture
        , ROW_NUMBER() OVER (PARTITION BY patientunitstayid ORDER BY culture_offset) AS me_id
    FROM cultures
)
, ab_fnl AS (
    SELECT
        ab.patientunitstayid
        , ab.ab_id
        , ab.antibiotic_offset
        , ab.antibiotic
        , me.me_id
        , me.culture_offset
        , me.specimen
        , me.positive_culture
        -- Suspected infection time: earlier of antibiotic or culture
        , LEAST(
            COALESCE(ab.antibiotic_offset, me.culture_offset),
            COALESCE(me.culture_offset, ab.antibiotic_offset)
          ) AS suspected_infection_time
        , 1 AS suspected_infection
    FROM ab_tbl ab
    LEFT JOIN me_tbl me
        ON ab.patientunitstayid = me.patientunitstayid
        -- Antibiotic within 24h before to 72h after culture
        AND ab.antibiotic_offset >= me.culture_offset - 1440  -- 24 hours before (in minutes)
        AND ab.antibiotic_offset <= me.culture_offset + 4320  -- 72 hours after (in minutes)
)
SELECT * FROM ab_fnl
WHERE suspected_infection = 1;

Query is running:   0%|          |

""


In [14]:
%%bigquery

-- Verification: Check updated suspicion_of_infection table
SELECT
    'suspicion_of_infection' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT patientunitstayid) as unique_icu_stays,
    SUM(positive_culture) as positive_cultures,
    COUNT(DISTINCT antibiotic) as unique_antibiotics
FROM `my-new-project-473015.my_eicu_derived.suspicion_of_infection`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,positive_cultures,unique_antibiotics
0,suspicion_of_infection,207882,61738,9909,82


---

## STEP 7: Create SOFA Score Table

This table calculates the Sequential Organ Failure Assessment (SOFA) score.

**SOFA Components:**
1. **Respiration:** PaO2/FiO2 ratio
2. **Coagulation:** Platelet count
3. **Liver:** Bilirubin
4. **Cardiovascular:** MAP and vasopressors
5. **CNS:** Glasgow Coma Scale
6. **Renal:** Creatinine and urine output

**Source:** `lab`, `vitalsign`, `medication`, `apacheApsVar`, `physicalexam` tables

In [15]:
%%bigquery

-- Recreate SOFA table with ABG/VBG filtering logic
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sofa` AS
WITH patient_info AS (
    SELECT
        patientunitstayid
        , uniquepid
        , CAST(unitdischargeoffset / 60 AS INT64) AS icu_los_hours
    FROM `physionet-data.eicu_crd.patient`
)
, hourly_grid AS (
    SELECT
        p.patientunitstayid
        , p.uniquepid
        , hr
        , hr * 60 AS startoffset
        , (hr + 1) * 60 AS endoffset
    FROM patient_info p
    CROSS JOIN UNNEST(GENERATE_ARRAY(0, LEAST(p.icu_los_hours, 168))) AS hr
)
-- Lab data from eicu_crd_derived.pivoted_lab
, lab_aggregated AS (
    SELECT
        hg.patientunitstayid
        , hg.hr
        , MIN(lb.platelets) AS platelet_min
        , MAX(lb.bilirubin) AS bilirubin_max
        , MAX(lb.creatinine) AS creatinine_max
    FROM hourly_grid hg
    LEFT JOIN `physionet-data.eicu_crd_derived.pivoted_lab` lb
        ON hg.patientunitstayid = lb.patientunitstayid
        AND lb.chartoffset >= hg.startoffset
        AND lb.chartoffset < hg.endoffset
    GROUP BY hg.patientunitstayid, hg.hr
)
-- ABG reference values from apacheApsVar for VBG exclusion
, apache_abg AS (
    SELECT
        patientunitstayid,
        pao2 AS apache_pao2
    FROM `physionet-data.eicu_crd.apacheapsvar`
)
-- Blood gas from eicu_crd_derived.pivoted_bg (with ABG filtering)
, bg_aggregated AS (
    SELECT
        hg.patientunitstayid
        , hg.hr
        , MIN(CASE
            -- Apply ABG filter only to PaO2
            WHEN a.apache_pao2 IS NULL OR bg.pao2 >= a.apache_pao2 * 0.9
            THEN bg.pao2
            ELSE NULL
          END) AS pao2_min
        , MAX(bg.fio2) AS fio2_max
    FROM hourly_grid hg
    LEFT JOIN `physionet-data.eicu_crd_derived.pivoted_bg` bg
        ON hg.patientunitstayid = bg.patientunitstayid
        AND bg.chartoffset >= hg.startoffset
        AND bg.chartoffset < hg.endoffset
        AND (bg.pao2 IS NOT NULL OR bg.fio2 IS NOT NULL)
    LEFT JOIN apache_abg a
        ON hg.patientunitstayid = a.patientunitstayid
    GROUP BY hg.patientunitstayid, hg.hr
)
-- GCS from eicu_crd_derived.pivoted_gcs
, gcs_aggregated AS (
    SELECT
        hg.patientunitstayid
        , hg.hr
        , MIN(CAST(gd.gcs AS FLOAT64)) AS gcs_min
    FROM hourly_grid hg
    LEFT JOIN `physionet-data.eicu_crd_derived.pivoted_gcs` gd
        ON hg.patientunitstayid = gd.patientunitstayid
        AND gd.chartoffset >= hg.startoffset
        AND gd.chartoffset < hg.endoffset
        AND gd.gcs IS NOT NULL
        AND gd.gcs >= 3
        AND gd.gcs <= 15
    GROUP BY hg.patientunitstayid, hg.hr
)
-- Vitals from eicu_crd_derived.pivoted_vital (use ibp_mean if available, otherwise nibp_mean)
, vital_aggregated AS (
    SELECT
        hg.patientunitstayid
        , hg.hr
        , MIN(COALESCE(vd.ibp_mean, vd.nibp_mean)) AS meanbp_min
    FROM hourly_grid hg
    LEFT JOIN `physionet-data.eicu_crd_derived.pivoted_vital` vd
        ON hg.patientunitstayid = vd.patientunitstayid
        AND vd.chartoffset >= hg.startoffset
        AND vd.chartoffset < hg.endoffset
        AND (vd.ibp_mean IS NOT NULL OR vd.nibp_mean IS NOT NULL)
    GROUP BY hg.patientunitstayid, hg.hr
)
-- Vasopressors from eicu_crd.infusiondrug
, vaso_data AS (
    SELECT
        patientunitstayid
        , infusionoffset
        , CASE WHEN LOWER(drugname) LIKE '%dopamine%'
               THEN SAFE_CAST(drugrate AS FLOAT64) END AS rate_dopamine
        , CASE WHEN LOWER(drugname) LIKE '%dobutamine%'
               THEN SAFE_CAST(drugrate AS FLOAT64) END AS rate_dobutamine
        , CASE WHEN LOWER(drugname) LIKE '%epinephrine%' AND LOWER(drugname) NOT LIKE '%norepinephrine%'
               THEN SAFE_CAST(drugrate AS FLOAT64) END AS rate_epinephrine
        , CASE WHEN LOWER(drugname) LIKE '%norepinephrine%' OR LOWER(drugname) LIKE '%levophed%'
               THEN SAFE_CAST(drugrate AS FLOAT64) END AS rate_norepinephrine
    FROM `physionet-data.eicu_crd.infusiondrug`
    WHERE infusionoffset >= 0
)
, vaso_aggregated AS (
    SELECT
        hg.patientunitstayid
        , hg.hr
        , MAX(vd.rate_dopamine) AS rate_dopamine
        , MAX(vd.rate_dobutamine) AS rate_dobutamine
        , MAX(vd.rate_epinephrine) AS rate_epinephrine
        , MAX(vd.rate_norepinephrine) AS rate_norepinephrine
    FROM hourly_grid hg
    LEFT JOIN vaso_data vd
        ON hg.patientunitstayid = vd.patientunitstayid
        AND vd.infusionoffset >= hg.startoffset
        AND vd.infusionoffset < hg.endoffset
    GROUP BY hg.patientunitstayid, hg.hr
)
-- Urine output from my_eicu_derived.urine_output_rate
, uo_aggregated AS (
    SELECT
        hg.patientunitstayid
        , hg.hr
        , MAX(
            CASE WHEN ud.uo_tm_24hr >= 22 AND ud.uo_tm_24hr <= 30
                THEN ud.uo_24hr / ud.uo_tm_24hr * 24
            END
        ) AS uo_24hr
    FROM hourly_grid hg
    LEFT JOIN `my-new-project-473015.my_eicu_derived.urine_output_rate` ud
        ON hg.patientunitstayid = ud.patientunitstayid
        AND ud.chartoffset >= hg.startoffset
        AND ud.chartoffset < hg.endoffset
    GROUP BY hg.patientunitstayid, hg.hr
)
-- Ventilation from my_eicu_derived.ventilation
, vent_aggregated AS (
    SELECT
        hg.patientunitstayid
        , hg.hr
        , MAX(CASE WHEN vd.ventilation_status = 'InvasiveVent' THEN 1 ELSE 0 END) AS vent
    FROM hourly_grid hg
    LEFT JOIN `my-new-project-473015.my_eicu_derived.ventilation` vd
        ON hg.patientunitstayid = vd.patientunitstayid
        AND vd.chartoffset >= hg.startoffset
        AND vd.chartoffset < hg.endoffset
    GROUP BY hg.patientunitstayid, hg.hr
)
, scorecomp AS (
    SELECT
        hg.patientunitstayid
        , hg.uniquepid
        , hg.hr
        , la.platelet_min
        , la.bilirubin_max
        , la.creatinine_max
        , bg.pao2_min
        , bg.fio2_max
        , va.meanbp_min
        , vs.rate_dopamine
        , vs.rate_dobutamine
        , vs.rate_epinephrine
        , vs.rate_norepinephrine
        , gc.gcs_min
        , uo.uo_24hr
        , COALESCE(vt.vent, 0) AS vent
        , CASE
            WHEN bg.fio2_max > 0 AND bg.pao2_min IS NOT NULL
            THEN bg.pao2_min / (CASE WHEN bg.fio2_max > 1 THEN bg.fio2_max / 100.0 ELSE bg.fio2_max END)
            ELSE NULL
          END AS pao2fio2ratio
    FROM hourly_grid hg
    LEFT JOIN lab_aggregated la ON hg.patientunitstayid = la.patientunitstayid AND hg.hr = la.hr
    LEFT JOIN bg_aggregated bg ON hg.patientunitstayid = bg.patientunitstayid AND hg.hr = bg.hr
    LEFT JOIN vital_aggregated va ON hg.patientunitstayid = va.patientunitstayid AND hg.hr = va.hr
    LEFT JOIN vaso_aggregated vs ON hg.patientunitstayid = vs.patientunitstayid AND hg.hr = vs.hr
    LEFT JOIN gcs_aggregated gc ON hg.patientunitstayid = gc.patientunitstayid AND hg.hr = gc.hr
    LEFT JOIN uo_aggregated uo ON hg.patientunitstayid = uo.patientunitstayid AND hg.hr = uo.hr
    LEFT JOIN vent_aggregated vt ON hg.patientunitstayid = vt.patientunitstayid AND hg.hr = vt.hr
)
, scorecomp_with_pf AS (
    SELECT
        scorecomp.*
        , CASE WHEN vent = 1 THEN pao2fio2ratio ELSE NULL END AS pao2fio2ratio_vent
        , CASE WHEN vent = 0 OR vent IS NULL THEN pao2fio2ratio ELSE NULL END AS pao2fio2ratio_novent
    FROM scorecomp
)
, scorecalc AS (
    SELECT scorecomp_with_pf.*
        -- Respiration SOFA
        , CASE
            WHEN pao2fio2ratio_vent < 100 THEN 4
            WHEN pao2fio2ratio_vent < 200 THEN 3
            WHEN pao2fio2ratio_novent < 300 THEN 2
            WHEN pao2fio2ratio_vent < 300 THEN 2
            WHEN pao2fio2ratio_novent < 400 THEN 1
            WHEN pao2fio2ratio_vent < 400 THEN 1
            WHEN COALESCE(pao2fio2ratio_vent, pao2fio2ratio_novent) IS NULL THEN NULL
            ELSE 0
        END AS respiration
        -- Coagulation SOFA
        , CASE
            WHEN platelet_min < 20 THEN 4
            WHEN platelet_min < 50 THEN 3
            WHEN platelet_min < 100 THEN 2
            WHEN platelet_min < 150 THEN 1
            WHEN platelet_min IS NULL THEN NULL
            ELSE 0
        END AS coagulation
        -- Liver SOFA
        , CASE
            WHEN bilirubin_max >= 12.0 THEN 4
            WHEN bilirubin_max >= 6.0 THEN 3
            WHEN bilirubin_max >= 2.0 THEN 2
            WHEN bilirubin_max >= 1.2 THEN 1
            WHEN bilirubin_max IS NULL THEN NULL
            ELSE 0
        END AS liver
        -- Cardiovascular SOFA
        , CASE
            WHEN rate_dopamine > 15 OR rate_epinephrine > 0.1 OR rate_norepinephrine > 0.1 THEN 4
            WHEN rate_dopamine > 5
                 OR (rate_epinephrine > 0 AND rate_epinephrine <= 0.1)
                 OR (rate_norepinephrine > 0 AND rate_norepinephrine <= 0.1) THEN 3
            WHEN rate_dopamine > 0 OR rate_dobutamine > 0 THEN 2
            WHEN meanbp_min < 70 THEN 1
            WHEN COALESCE(meanbp_min, rate_dopamine, rate_dobutamine, rate_epinephrine, rate_norepinephrine) IS NULL THEN NULL
            ELSE 0
        END AS cardiovascular
        -- CNS SOFA
        , CASE
            WHEN gcs_min >= 13 AND gcs_min <= 14 THEN 1
            WHEN gcs_min >= 10 AND gcs_min <= 12 THEN 2
            WHEN gcs_min >= 6 AND gcs_min <= 9 THEN 3
            WHEN gcs_min < 6 THEN 4
            WHEN gcs_min IS NULL THEN NULL
            ELSE 0
        END AS cns
        -- Renal SOFA
        , CASE
            WHEN creatinine_max >= 5.0 THEN 4
            WHEN uo_24hr < 200 THEN 4
            WHEN creatinine_max >= 3.5 AND creatinine_max < 5.0 THEN 3
            WHEN uo_24hr < 500 THEN 3
            WHEN creatinine_max >= 2.0 AND creatinine_max < 3.5 THEN 2
            WHEN creatinine_max >= 1.2 AND creatinine_max < 2.0 THEN 1
            WHEN COALESCE(uo_24hr, creatinine_max) IS NULL THEN NULL
            ELSE 0
        END AS renal
    FROM scorecomp_with_pf
)
, score_final AS (
    SELECT s.*
        , COALESCE(MAX(respiration) OVER w, 0) AS respiration_24hours
        , COALESCE(MAX(coagulation) OVER w, 0) AS coagulation_24hours
        , COALESCE(MAX(liver) OVER w, 0) AS liver_24hours
        , COALESCE(MAX(cardiovascular) OVER w, 0) AS cardiovascular_24hours
        , COALESCE(MAX(cns) OVER w, 0) AS cns_24hours
        , COALESCE(MAX(renal) OVER w, 0) AS renal_24hours
        , COALESCE(MAX(respiration) OVER w, 0)
        + COALESCE(MAX(coagulation) OVER w, 0)
        + COALESCE(MAX(liver) OVER w, 0)
        + COALESCE(MAX(cardiovascular) OVER w, 0)
        + COALESCE(MAX(cns) OVER w, 0)
        + COALESCE(MAX(renal) OVER w, 0) AS sofa_24hours
    FROM scorecalc s
    WINDOW w AS (
        PARTITION BY patientunitstayid
        ORDER BY hr
        ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
    )
)
SELECT * FROM score_final
WHERE hr >= 0;

Query is running:   0%|          |

""


In [16]:
%%bigquery

-- Verification: Check data completeness after using all derived tables
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT patientunitstayid) AS unique_stays,
    SUM(CASE WHEN pao2fio2ratio IS NOT NULL THEN 1 ELSE 0 END) AS has_pf_ratio,
    SUM(CASE WHEN platelet_min IS NOT NULL THEN 1 ELSE 0 END) AS has_platelet,
    SUM(CASE WHEN bilirubin_max IS NOT NULL THEN 1 ELSE 0 END) AS has_bilirubin,
    SUM(CASE WHEN creatinine_max IS NOT NULL THEN 1 ELSE 0 END) AS has_creatinine,
    SUM(CASE WHEN meanbp_min IS NOT NULL THEN 1 ELSE 0 END) AS has_mbp,
    SUM(CASE WHEN gcs_min IS NOT NULL THEN 1 ELSE 0 END) AS has_gcs,
    SUM(CASE WHEN uo_24hr IS NOT NULL THEN 1 ELSE 0 END) AS has_uo
FROM `my-new-project-473015.my_eicu_derived.sofa`
WHERE hr = 24;

Query is running:   0%|          |

Downloading:   0%|          |

,total_rows,unique_stays,has_pf_ratio,has_platelet,has_bilirubin,has_creatinine,has_mbp,has_gcs,has_uo
0,134884,134884,1123,4140,1360,4961,85382,26356,19623


In [17]:
%%bigquery

-- Verification: Check SOFA table
SELECT
    'sofa' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT patientunitstayid) as unique_icu_stays,
    ROUND(AVG(sofa_24hours), 2) as avg_sofa,
    MIN(sofa_24hours) as min_sofa,
    MAX(sofa_24hours) as max_sofa
FROM `my-new-project-473015.my_eicu_derived.sofa`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_sofa,min_sofa,max_sofa
0,sofa,10898549,200857,3.21,0,23


---

## STEP 8: Create Sepsis-3 Table

This table identifies patients meeting Sepsis-3 criteria:
1. **Suspected infection** (antibiotics + cultures within time window)
2. **SOFA score ≥ 2** from baseline

**Output:** One row per ICU stay with sepsis3 flag

In [18]:
%%bigquery

-- Recreate sepsis3 table
CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3` AS
WITH sofa_filtered AS (
    SELECT
        patientunitstayid
        , uniquepid
        , hr
        , hr * 60 AS hr_offset
        , respiration_24hours AS respiration
        , coagulation_24hours AS coagulation
        , liver_24hours AS liver
        , cardiovascular_24hours AS cardiovascular
        , cns_24hours AS cns
        , renal_24hours AS renal
        , sofa_24hours AS sofa_score
    FROM `my-new-project-473015.my_eicu_derived.sofa`
    WHERE sofa_24hours >= 2
)
, s1 AS (
    SELECT
        soi.patientunitstayid
        , sf.uniquepid
        , soi.ab_id
        , soi.antibiotic
        , soi.antibiotic_offset
        , soi.culture_offset
        , soi.suspected_infection
        , soi.suspected_infection_time
        , soi.specimen
        , soi.positive_culture
        , sf.hr
        , sf.hr_offset AS sofa_offset
        , sf.respiration
        , sf.coagulation
        , sf.liver
        , sf.cardiovascular
        , sf.cns
        , sf.renal
        , sf.sofa_score
        , sf.sofa_score >= 2 AND soi.suspected_infection = 1 AS sepsis3
        , ROW_NUMBER() OVER (
            PARTITION BY soi.patientunitstayid
            ORDER BY soi.suspected_infection_time,
                     soi.antibiotic_offset,
                     soi.culture_offset,
                     sf.hr_offset
        ) AS rn_sus
    FROM `my-new-project-473015.my_eicu_derived.suspicion_of_infection` AS soi
    INNER JOIN sofa_filtered sf
        ON soi.patientunitstayid = sf.patientunitstayid
        AND sf.hr_offset >= soi.suspected_infection_time - 2880
        AND sf.hr_offset <= soi.suspected_infection_time + 1440
    WHERE soi.patientunitstayid IS NOT NULL
)
SELECT
    patientunitstayid
    , uniquepid
    , antibiotic_offset
    , culture_offset
    , suspected_infection_time
    , sofa_offset AS sofa_time
    , sofa_score
    , respiration
    , coagulation
    , liver
    , cardiovascular
    , cns
    , renal
    , sepsis3
FROM s1
WHERE rn_sus = 1;

Query is running:   0%|          |

""


In [19]:
%%bigquery

-- Verification: Check sepsis3 table structure
SELECT column_name, data_type
FROM `my-new-project-473015.my_eicu_derived.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'sepsis3'
ORDER BY ordinal_position;

Query is running:   0%|          |

Downloading:   0%|          |

,column_name,data_type
0,patientunitstayid,INT64
1,uniquepid,STRING
2,antibiotic_offset,INT64
3,culture_offset,INT64
4,suspected_infection_time,INT64
5,sofa_time,INT64
6,sofa_score,INT64
7,respiration,INT64
8,coagulation,INT64
9,liver,INT64


In [20]:
%%bigquery

-- Compare with total eICU population
WITH sepsis_stats AS (
    SELECT
        COUNT(DISTINCT patientunitstayid) AS sepsis3_icu_stays,
        COUNT(DISTINCT patientunitstayid) AS sepsis3_patients
    FROM `my-new-project-473015.my_eicu_derived.sepsis3`
    WHERE sepsis3 = TRUE
),
total_stats AS (
    SELECT COUNT(DISTINCT patientunitstayid) AS total_icu_stays
    FROM `physionet-data.eicu_crd.patient`
)
SELECT
    s.sepsis3_icu_stays,
    t.total_icu_stays,
    ROUND(100.0 * s.sepsis3_icu_stays / t.total_icu_stays, 1) AS sepsis3_percentage
FROM sepsis_stats s, total_stats t;

Query is running:   0%|          |

Downloading:   0%|          |

,sepsis3_icu_stays,total_icu_stays,sepsis3_percentage
0,43572,200859,21.7


---

## Summary: All Tables Created

The following tables have been created in `my-new-project-473015.my_eicu_derived`:

| Table | Description | Key Columns |
|-------|-------------|-------------|
| vitalsign | Vital signs from vitalPeriodic/Aperiodic | patientunitstayid, observationoffset, heart_rate, sbp, dbp, mbp, resp_rate, spo2, temperature |
| weight_icustay | Patient weights | patientunitstayid, weight |
| urine_output | Urine output measurements | patientunitstayid, chartoffset, urineoutput |
| urine_output_rate | Urine output rates (24hr) | patientunitstayid, chartoffset, uo_24hr, uo_mlkghr_24hr |
| ventilation | Ventilation status | patientunitstayid, chartoffset, ventilation_status |
| suspicion_of_infection | Suspected infections | patientunitstayid, suspected_infection_time, antibiotic, positive_culture |
| sofa | SOFA scores (hourly) | patientunitstayid, hr, sofa_24hours, component scores |
| sepsis3 | Sepsis-3 classification | patientunitstayid, uniquepid, sepsis3, sofa_score |

---

## Key Differences from MIMIC-IV Implementation

1. **Patient Identifiers:**
   - eICU: `patientunitstayid` (ICU stay), `uniquepid` (patient)
   - MIMIC-IV: `stay_id` (ICU stay), `subject_id` (patient)

2. **Time Representation:**
   - eICU: Offset in minutes from ICU admission (e.g., `observationoffset`)
   - MIMIC-IV: Absolute timestamps (e.g., `charttime`)

3. **Data Sources:**
   - Vitals: eICU uses `vitalPeriodic`/`vitalAperiodic` vs MIMIC-IV's `chartevents`
   - Labs: eICU uses `lab` table vs MIMIC-IV's `labevents`
   - Medications: eICU uses `medication`/`infusionDrug` vs MIMIC-IV's `inputevents`/`emar`

4. **Temperature Units:**
   - eICU: Stored in Celsius
   - MIMIC-IV: Mix of Fahrenheit and Celsius (requires conversion)

In [21]:
%%bigquery

-- Final Summary: Compare with overall ICU population
WITH total_stats AS (
    SELECT
        COUNT(DISTINCT patientunitstayid) AS total_icu_stays,
        COUNT(DISTINCT uniquepid) AS total_patients
    FROM `physionet-data.eicu_crd.patient`
)
, sepsis_stats AS (
    SELECT
        COUNT(DISTINCT patientunitstayid) AS sepsis3_icu_stays,
        COUNT(DISTINCT uniquepid) AS sepsis3_patients,
        ROUND(AVG(sofa_score), 2) AS avg_sofa_score,
        MIN(sofa_score) AS min_sofa,
        MAX(sofa_score) AS max_sofa
    FROM `my-new-project-473015.my_eicu_derived.sepsis3`
    WHERE sepsis3 = TRUE
)
SELECT
    t.total_icu_stays,
    t.total_patients,
    s.sepsis3_icu_stays,
    s.sepsis3_patients,
    s.avg_sofa_score,
    s.min_sofa,
    s.max_sofa,
    ROUND(100.0 * s.sepsis3_icu_stays / t.total_icu_stays, 1) AS sepsis3_icu_pct,
    ROUND(100.0 * s.sepsis3_patients / t.total_patients, 1) AS sepsis3_patient_pct
FROM total_stats t, sepsis_stats s;

Query is running:   0%|          |

Downloading:   0%|          |

,total_icu_stays,total_patients,sepsis3_icu_stays,sepsis3_patients,avg_sofa_score,min_sofa,max_sofa,sepsis3_icu_pct,sepsis3_patient_pct
0,200859,139367,43572,36758,3.52,2,18,21.7,26.4


In [22]:
%%bigquery

SELECT
    'sofa' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT patientunitstayid) AS unique_icu_stays,
    COUNT(DISTINCT uniquepid) AS unique_patients,
    MIN(hr) AS min_hours_from_icu_admit,
    MAX(hr) AS max_hours_from_icu_admit
FROM `my-new-project-473015.my_eicu_derived.sofa`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,total_rows,unique_icu_stays,unique_patients,min_hours_from_icu_admit,max_hours_from_icu_admit
0,sofa,10898549,200857,139367,0,168


In [23]:
%%bigquery

-- 2. SOFA Score Distribution
SELECT
    sofa_24hours,
    COUNT(*) AS count,
    COUNT(DISTINCT patientunitstayid) AS unique_stays
FROM `my-new-project-473015.my_eicu_derived.sofa`
WHERE hr = 24  -- At 24 hours
GROUP BY sofa_24hours
ORDER BY sofa_24hours;

Query is running:   0%|          |

Downloading:   0%|          |

,sofa_24hours,count,unique_stays
0,0,17228,17228
1,1,26981,26981
2,2,19795,19795
3,3,15990,15990
4,4,13784,13784
5,5,11041,11041
6,6,8392,8392
7,7,6326,6326
8,8,4651,4651
9,9,3463,3463


In [24]:
%%bigquery

-- 3. SOFA Component Statistics (at hour 24)
SELECT
    'Hour 24 Statistics' AS timepoint,
    COUNT(*) AS n_records,
    -- Respiration
    ROUND(AVG(respiration_24hours), 2) AS avg_resp,
    SUM(CASE WHEN respiration_24hours IS NOT NULL THEN 1 ELSE 0 END) AS n_resp,
    -- Coagulation
    ROUND(AVG(coagulation_24hours), 2) AS avg_coag,
    SUM(CASE WHEN coagulation_24hours IS NOT NULL THEN 1 ELSE 0 END) AS n_coag,
    -- Liver
    ROUND(AVG(liver_24hours), 2) AS avg_liver,
    SUM(CASE WHEN liver_24hours IS NOT NULL THEN 1 ELSE 0 END) AS n_liver,
    -- Cardiovascular
    ROUND(AVG(cardiovascular_24hours), 2) AS avg_cardio,
    SUM(CASE WHEN cardiovascular_24hours IS NOT NULL THEN 1 ELSE 0 END) AS n_cardio,
    -- CNS
    ROUND(AVG(cns_24hours), 2) AS avg_cns,
    SUM(CASE WHEN cns_24hours IS NOT NULL THEN 1 ELSE 0 END) AS n_cns,
    -- Renal
    ROUND(AVG(renal_24hours), 2) AS avg_renal,
    SUM(CASE WHEN renal_24hours IS NOT NULL THEN 1 ELSE 0 END) AS n_renal,
    -- Total SOFA
    ROUND(AVG(sofa_24hours), 2) AS avg_sofa_total,
    MIN(sofa_24hours) AS min_sofa,
    MAX(sofa_24hours) AS max_sofa
FROM `my-new-project-473015.my_eicu_derived.sofa`
WHERE hr = 24;

Query is running:   0%|          |

Downloading:   0%|          |

,timepoint,n_records,avg_resp,n_resp,avg_coag,n_coag,avg_liver,n_liver,avg_cardio,n_cardio,avg_cns,n_cns,avg_renal,n_renal,avg_sofa_total,min_sofa,max_sofa
0,Hour 24 Statistics,134884,0.43,134884,0.41,134884,0.16,134884,0.99,134884,0.85,134884,0.68,134884,3.5,0,22


In [25]:
%%bigquery

-- 4. Data Completeness by Component (first 24 hours)
SELECT
    'First 24h' AS period,
    COUNT(DISTINCT patientunitstayid) AS total_stays,
    -- Raw component availability
    COUNT(DISTINCT CASE WHEN pao2fio2ratio IS NOT NULL THEN patientunitstayid END) AS has_pf_ratio,
    COUNT(DISTINCT CASE WHEN platelet_min IS NOT NULL THEN patientunitstayid END) AS has_platelet,
    COUNT(DISTINCT CASE WHEN bilirubin_max IS NOT NULL THEN patientunitstayid END) AS has_bilirubin,
    COUNT(DISTINCT CASE WHEN meanbp_min IS NOT NULL THEN patientunitstayid END) AS has_mbp,
    COUNT(DISTINCT CASE WHEN gcs_min IS NOT NULL THEN patientunitstayid END) AS has_gcs,
    COUNT(DISTINCT CASE WHEN creatinine_max IS NOT NULL THEN patientunitstayid END) AS has_creatinine,
    COUNT(DISTINCT CASE WHEN uo_24hr IS NOT NULL THEN patientunitstayid END) AS has_urine_output,
    COUNT(DISTINCT CASE WHEN vent IS NOT NULL THEN patientunitstayid END) AS has_vent_status
FROM `my-new-project-473015.my_eicu_derived.sofa`
WHERE hr BETWEEN 0 AND 24;

Query is running:   0%|          |

Downloading:   0%|          |

,period,total_stays,has_pf_ratio,has_platelet,has_bilirubin,has_mbp,has_gcs,has_creatinine,has_urine_output,has_vent_status
0,First 24h,200857,47147,152635,70636,178977,141120,161340,38606,200857


In [26]:
%%bigquery

-- 5. Sample Data: First 10 patients at hour 24
SELECT
    patientunitstayid,
    uniquepid,
    hr,
    -- Raw values
    ROUND(pao2fio2ratio, 1) AS pf_ratio,
    platelet_min,
    ROUND(bilirubin_max, 1) AS bilirubin,
    ROUND(creatinine_max, 1) AS creatinine,
    ROUND(meanbp_min, 0) AS mbp_min,
    gcs_min,
    ROUND(uo_24hr, 0) AS uo_24hr,
    vent,
    -- Component scores
    respiration_24hours AS resp_score,
    coagulation_24hours AS coag_score,
    liver_24hours AS liver_score,
    cardiovascular_24hours AS cardio_score,
    cns_24hours AS cns_score,
    renal_24hours AS renal_score,
    sofa_24hours AS total_sofa
FROM `my-new-project-473015.my_eicu_derived.sofa`
WHERE hr = 24
ORDER BY sofa_24hours DESC
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,patientunitstayid,uniquepid,hr,pf_ratio,platelet_min,bilirubin,creatinine,mbp_min,gcs_min,uo_24hr,vent,resp_score,coag_score,liver_score,cardio_score,cns_score,renal_score,total_sofa
0,979130,007-10833,24,NaN,NaN,NaN,NaN,90.0,3.0,0.0,1,2,4,4,4,4,4,22
1,1097725,010-28195,24,168.0,NaN,NaN,NaN,64.0,NaN,145.0,0,2,4,3,4,4,4,21
2,945476,006-116346,24,NaN,NaN,NaN,NaN,60.0,NaN,NaN,0,2,3,4,4,4,4,21
3,3053979,030-41803,24,76.0,NaN,NaN,NaN,47.0,3.0,0.0,0,4,3,2,4,4,4,21
4,3036868,030-56934,24,NaN,NaN,NaN,NaN,74.0,NaN,NaN,0,2,2,4,4,4,4,20
5,1130744,010-30106,24,NaN,12.0,NaN,NaN,64.0,10.0,NaN,0,2,4,2,4,4,4,20
6,1135058,010-41299,24,NaN,NaN,NaN,NaN,73.0,NaN,194.0,0,2,2,4,4,4,4,20
7,934132,006-50478,24,122.1,NaN,NaN,NaN,65.0,3.0,NaN,0,2,3,3,4,4,4,20
8,1106921,010-21438,24,NaN,NaN,NaN,NaN,41.0,3.0,496.0,0,2,4,3,4,4,3,20
9,1112386,010-30268,24,NaN,NaN,NaN,NaN,63.0,NaN,174.0,0,2,3,3,4,4,4,20


In [27]:
%%bigquery

-- 6. Time Series Sample: Single Patient SOFA Trajectory
WITH sample_patient AS (
    SELECT patientunitstayid
    FROM `my-new-project-473015.my_eicu_derived.sofa`
    WHERE hr = 24 AND sofa_24hours >= 5
    LIMIT 1
)
SELECT
    s.patientunitstayid,
    s.hr,
    s.respiration_24hours,
    s.coagulation_24hours,
    s.liver_24hours,
    s.cardiovascular_24hours,
    s.cns_24hours,
    s.renal_24hours,
    s.sofa_24hours
FROM `my-new-project-473015.my_eicu_derived.sofa` s
INNER JOIN sample_patient sp ON s.patientunitstayid = sp.patientunitstayid
WHERE s.hr BETWEEN 0 AND 72
ORDER BY s.hr;

Query is running:   0%|          |

Downloading:   0%|          |

,patientunitstayid,hr,respiration_24hours,coagulation_24hours,liver_24hours,cardiovascular_24hours,cns_24hours,renal_24hours,sofa_24hours
0,1240634,0,0,0,0,0,0,0,0
1,1240634,1,0,0,0,0,0,0,0
2,1240634,2,0,0,0,0,0,0,0
3,1240634,3,0,0,2,0,0,4,6
4,1240634,4,0,0,2,0,0,4,6
...,...,...,...,...,...,...,...,...,...
68,1240634,68,2,0,3,1,0,4,10
69,1240634,69,2,0,3,1,0,4,10
70,1240634,70,2,0,3,1,0,4,10
71,1240634,71,2,0,3,1,0,4,10


In [28]:
%%bigquery

-- 7. Schema/Column Information
SELECT
    column_name,
    data_type,
    is_nullable
FROM `my-new-project-473015.my_eicu_derived.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'sofa'
ORDER BY ordinal_position;

Query is running:   0%|          |

Downloading:   0%|          |

,column_name,data_type,is_nullable
0,patientunitstayid,INT64,YES
1,uniquepid,STRING,YES
2,hr,INT64,YES
3,platelet_min,FLOAT64,YES
4,bilirubin_max,FLOAT64,YES
5,creatinine_max,FLOAT64,YES
6,pao2_min,FLOAT64,YES
7,fio2_max,FLOAT64,YES
8,meanbp_min,FLOAT64,YES
9,rate_dopamine,FLOAT64,YES


## Key Differences from MIMIC-IV Implementation: Detailed Comparison

---

### 1. Database Overview

| Aspect | MIMIC-IV | eICU |
|--------|----------|------|
| Institution | Beth Israel Deaconess Medical Center (Boston) | 208 hospitals across the US |
| Time Period | 2008-2019 | 2014-2015 |
| Total ICU stays | ~94,000 | ~200,000 |
| Data Structure | Single-center, detailed | Multi-center, heterogeneous |
| De-identification | Date shifting | Date shifting + hospital anonymization |

**⚠️ Concern:** eICU is a multi-center database, so there may be variability in data recording methods and completeness across different facilities.

---

### 2. Patient Identifiers

| Purpose | MIMIC-IV | eICU |
|---------|----------|------|
| Patient ID | `subject_id` | `uniquepid` |
| Hospital admission ID | `hadm_id` | (Not directly available) |
| ICU stay ID | `stay_id` | `patientunitstayid` |

**⚠️ Concerns:**
- eICU lacks a `hadm_id` equivalent, making it difficult to track multiple ICU stays within the same hospital admission
- `uniquepid` is not a cross-institutional ID; the same patient admitted to different hospitals will have different IDs

---

### 3. Time Representation

| Aspect | MIMIC-IV | eICU |
|--------|----------|------|
| Format | Absolute datetime (`charttime`) | Offset in minutes (`observationoffset`) |
| Reference point | Calendar date/time | ICU admission = 0 |
| Pre-ICU data | Available (negative offsets possible) | Limited (negative offsets = pre-ICU) |
| Precision | Datetime | Integer minutes |

**Example:**
```
MIMIC-IV: charttime = '2150-03-15 14:30:00'
eICU:     observationoffset = 120 (= ICU admission + 2 hours)
```

**⚠️ Concerns:**
- In eICU, it is difficult to track time from hospital admission to ICU admission
- Interpretation of "time of suspected infection" in Sepsis-3 may differ between databases
- eICU tends to have less data with negative offsets (pre-ICU admission) compared to MIMIC-IV

---

### 4. Data Sources by Component

#### 4.1 Vital Signs

| Variable | MIMIC-IV Source | eICU Source | Notes |
|----------|-----------------|-------------|-------|
| Heart Rate | `chartevents` (itemid: 220045) | `vitalPeriodic.heartrate` | |
| SBP (invasive) | `chartevents` (220050, 220179) | `vitalPeriodic.systemicsystolic` | |
| SBP (non-invasive) | `chartevents` (220179) | `vitalAperiodic.noninvasivesystolic` | |
| DBP | `chartevents` (220051, 220180) | `vitalPeriodic.systemicdiastolic` | |
| MAP | `chartevents` (220052, 220181) | `vitalPeriodic.systemicmean` | |
| Respiratory Rate | `chartevents` (220210, 224690) | `vitalPeriodic.respiration` | |
| SpO2 | `chartevents` (220277) | `vitalPeriodic.sao2` | |
| Temperature | `chartevents` (223761=°F, 223762=°C) | `vitalPeriodic.temperature` | Unit difference |

**⚠️ Concerns:**
- MIMIC-IV stores all vitals in `chartevents` (distinguished by itemid)
- eICU separates data into `vitalPeriodic` (automatic recording every 5 minutes) and `vitalAperiodic` (irregular manual recording)
- **eICU's `vitalPeriodic` contains mostly automated monitor recordings, and missing data patterns may differ from MIMIC-IV**

#### 4.2 Laboratory Values

| Variable | MIMIC-IV Source | eICU Source | Notes |
|----------|-----------------|-------------|-------|
| Platelet | `labevents` (itemid: 51265) | `lab` (labname LIKE '%platelet%') | |
| Bilirubin | `labevents` (50885) | `lab` (labname LIKE '%bilirubin%total%') | |
| Creatinine | `labevents` (50912) | `lab` (labname LIKE '%creatinine%') | |
| PaO2 | `labevents` (50821) | `lab` (labname LIKE '%pao2%') | |
| FiO2 | `chartevents` / `labevents` | `lab` (labname LIKE '%fio2%') | Different sources |
| Lactate | `labevents` (50813) | `lab` (labname LIKE '%lactate%') | |

**⚠️ Concerns:**
- **MIMIC-IV can precisely identify variables using `itemid`, while eICU relies on text search of `labname`, introducing ambiguity**
- Lab names in eICU may vary across facilities (e.g., "Platelet Count" vs "Platelets" vs "PLT")
- Lab value units may not be standardized across facilities

#### 4.3 Glasgow Coma Scale (GCS)

| Component | MIMIC-IV Source | eICU Source |
|-----------|-----------------|-------------|
| GCS Total | `chartevents` (220739) or calculated | `physicalExam` (physicalExamPath LIKE '%gcs%total%') |
| GCS Eye | `chartevents` (220739) | `physicalExam` |
| GCS Verbal | `chartevents` (223900) | `physicalExam` |
| GCS Motor | `chartevents` (223901) | `physicalExam` |

**⚠️ Concerns:**
- eICU's `physicalExam` table tends to have lower GCS documentation rates than MIMIC-IV
- **When GCS is missing, CNS SOFA score may be underestimated**
- In eICU, GCS may also be recorded in the `nurseCharting` table, and both should be checked

#### 4.4 Urine Output

| Aspect | MIMIC-IV Source | eICU Source |
|--------|-----------------|-------------|
| Table | `outputevents` | `intakeOutput` |
| Filter | itemid IN (specific IDs) | celllabel LIKE '%urine%' |
| Negative values | GU irrigant (itemid 227488) | Less common |

**⚠️ Concerns:**
- eICU's `intakeOutput` uses free-text labels that vary across facilities
- **Classification accuracy for urine output categories may be lower than MIMIC-IV**

#### 4.5 Vasopressors

| Drug | MIMIC-IV Source | eICU Source | Unit Concern |
|------|-----------------|-------------|--------------|
| Norepinephrine | `inputevents` (221906) | `infusionDrug` (drugname LIKE '%norepinephrine%') | mcg/kg/min vs mcg/min |
| Epinephrine | `inputevents` (221289) | `infusionDrug` (drugname LIKE '%epinephrine%') | Same as above |
| Dopamine | `inputevents` (221662) | `infusionDrug` (drugname LIKE '%dopamine%') | mcg/kg/min |
| Dobutamine | `inputevents` (221653) | `infusionDrug` (drugname LIKE '%dobutamine%') | mcg/kg/min |

**⚠️ Major Concern:**
- **eICU's `drugrate` units are likely not standardized**
- Different facilities may record in mcg/kg/min, mcg/min, ml/hr, etc.
- **SOFA cardiovascular score thresholds (e.g., norepinephrine > 0.1 mcg/kg/min) may not be accurately applicable**
- MIMIC-IV's `inputevents` has a `rateuom` column to verify units

#### 4.6 Mechanical Ventilation

| Aspect | MIMIC-IV Source | eICU Source |
|--------|-----------------|-------------|
| Primary table | `ventilator_setting` + `oxygen_delivery` | `respiratoryCare` + `respiratoryCharting` |
| Ventilator mode | Detailed (CMV, SIMV, PSV, etc.) | Less standardized |
| Intubation status | Derived from device type | `apacheApsVar.intubated` flag available |

**⚠️ Concerns:**
- eICU ventilator information is scattered across `respiratoryCare.airwayType` and `respiratoryCharting`
- **Mechanical ventilation detection accuracy may be lower than MIMIC-IV**
- Respiratory SOFA score may be affected for conditions requiring PaO2/FiO2 < 200 + mechanical ventilation

#### 4.7 Suspicion of Infection

| Component | MIMIC-IV Source | eICU Source |
|-----------|-----------------|-------------|
| Antibiotics | `prescriptions` + `pharmacy` | `medication` |
| Cultures | `microbiologyevents` | `microlab` |
| Culture timing | `chartdate` / `charttime` | `cultureTakenOffset` |

**⚠️ Concerns:**
- eICU's `medication` table contains prescription information, while MIMIC-IV includes administration information
- **Definition of antibiotic "start time" may differ between databases**
- In eICU, it may be difficult to distinguish antibiotic routes (IV vs PO)

---

### 5. Units and Measurements

| Variable | MIMIC-IV | eICU | Conversion Needed |
|----------|----------|------|-------------------|
| Temperature | °F and °C mixed | °C | MIMIC: (°F - 32) / 1.8 |
| Creatinine | mg/dL | mg/dL | No |
| Bilirubin | mg/dL | mg/dL | No |
| Platelet | K/uL (×10³/μL) | K/uL | No |
| PaO2 | mmHg | mmHg | No |
| FiO2 | Fraction (0-1) or % (21-100) | Variable | Standardization needed |
| Weight | kg | kg | No |
| Urine output | mL | mL | No |
| Vasopressor rates | mcg/kg/min (standardized) | Variable | **Major concern** |

---

### 6. Sepsis-3 Criteria Implementation Differences

#### 6.1 Suspected Infection Definition

| Criterion | MIMIC-IV Implementation | eICU Implementation |
|-----------|------------------------|---------------------|
| Antibiotic + Culture window | Abx within 72h after culture OR culture within 24h before Abx | Same logic attempted |
| Antibiotic source | First IV antibiotic from `emar`/`prescriptions` | First antibiotic from `medication` |
| Culture source | `microbiologyevents.chartdate` | `microlab.cultureTakenOffset` |

**⚠️ Concerns:**
- In eICU, it is difficult to distinguish between oral and intravenous antibiotics
- **Definition of suspected infection patients may be broader (increased false positives)**

#### 6.2 SOFA Score Calculation

| Aspect | MIMIC-IV | eICU | Impact |
|--------|----------|------|--------|
| Baseline SOFA | Hospital admission or pre-infection | First 6 hours of ICU | Different baseline definition |
| Time window | 48h before to 24h after infection | Same | |
| Missing data handling | Component = NULL → excluded from sum | Same | |

**⚠️ Concerns:**
- **Baseline SOFA definition differs between databases**
- MIMIC-IV can calculate SOFA at hospital admission; eICU can only calculate at ICU admission
- This affects the determination of SOFA increase (≥2)

---

### 7. Data Quality and Completeness Concerns

| Issue | MIMIC-IV | eICU | Research Impact |
|-------|----------|------|-----------------|
| Missing vitals | Low (frequent charting) | Variable by hospital | SOFA calculation affected |
| Missing labs | Moderate | Higher in some facilities | Component scores affected |
| GCS documentation | High | Lower | CNS SOFA underestimated |
| Vasopressor units | Standardized | Not standardized | Cardiovascular SOFA unreliable |
| Ventilator status | Well-documented | Less complete | Respiratory SOFA affected |

---

### 8. Recommendations for Cross-Database Analysis

#### 8.1 Data Harmonization Steps

1. **Vasopressor rates**: In eICU, consider evaluating cardiovascular SOFA using MAP < 70 only, or use vasopressor administration as binary (yes/no)
2. **GCS**: In eICU, additionally check `nurseCharting` table
3. **FiO2**: Standardize values of 21-100 by dividing by 100 to convert to 0-1 scale
4. **Ventilation**: Also reference `apacheApsVar.intubated`

#### 8.2 Sensitivity Analysis Recommendations
```
1. Vasopressor impact sensitivity analysis
   - Cardiovascular SOFA = MAP-based only vs full criteria
   
2. GCS missing data sensitivity analysis
   - GCS missing → CNS SOFA = 0 vs exclude
   
3. Suspected infection definition sensitivity analysis
   - Antibiotics: IV only vs all routes
   - Cultures: Blood culture only vs all sites
```

#### 8.3 Reporting Recommendations

The following should be clearly stated in publications:
- Detailed variable extraction methods for each database
- Proportion and handling of missing data
- Limitations regarding vasopressor units
- Handling of inter-facility variability (eICU)

---

### 9. Summary Table: Critical Concerns

| Concern Level | Issue | Affected SOFA Component | Mitigation |
|---------------|-------|------------------------|------------|
| 🔴 Critical | Vasopressor unit inconsistency | Cardiovascular | Use binary (yes/no) or MAP only |
| 🟠 High | GCS documentation rate | CNS | Include `nurseCharting`, sensitivity analysis |
| 🟠 High | Ventilator status accuracy | Respiratory | Use `apacheApsVar.intubated` as backup |
| 🟡 Medium | Lab name text matching | All lab-based | Validate with frequency analysis |
| 🟡 Medium | Antibiotic route distinction | Infection definition | IV-only sensitivity analysis |
| 🟢 Low | Temperature units | (Not in SOFA) | Conversion applied |